# 11 — Pull Africa Leadership Change (ALC) Dataset

**Source:** Africa Leadership Change (ALC) Project  
**Provider:** Giovanni Carbone & Alessandro Pellegata (ISPI / University of Milan)  
**Project page:** https://www.ispionline.it/en/publication/africa-leadership-change-project-21287  
**DataLab:** https://alcafricandatalab.com/  
**Access:** Free download from the ISPI project page (registration may be required)  
**Coverage:** All 54 sub-Saharan African countries, from independence (~1960) to present  
**Frequency:** Leader-spell level (entry/exit per leader per country)  

## What this notebook pulls

The ALC dataset, created by Giovanni Carbone and Alessandro Pellegata at ISPI and
the University of Milan, provides systematic coding of African leadership transitions
including the mode of entry to and exit from power, regime type, and tenure duration.
It complements Archigos (global coverage) with Africa-specific detail and is used
alongside Powell-Thyne coup data for Africa-focused governance models.

### Key variables

| Variable | Description |
|---|---|
| `country` | Country name |
| `ccode` | Correlates of War country code |
| `leader` | Leader name |
| `startdate` / `start_year` | Date/year entered office |
| `enddate` / `end_year` | Date/year left office |
| `entry_type` | How the leader came to power (election, coup, inheritance, etc.) |
| `exit_type` | How the leader left (regular, coup, death, term limit, etc.) |
| `irregular_entry` | Irregular entry to power (0/1) |
| `irregular_exit` | Irregular removal from office (0/1) |
| `military_regime` | Country under military rule during spell (0/1) |

### Derived country-year panel variables
- `is_leader_change` — any leadership change in country-year
- `irregular_entry_count` — irregular entries to power
- `irregular_exit_count` — irregular exits from power
- `n_leaders` — number of distinct leaders active this year
- `any_military_regime` — military regime for any part of the year

## Access note
Download the dataset from the ISPI project page:
  https://www.ispionline.it/en/publication/africa-leadership-change-project-21287

An interactive DataLab is also available at https://alcafricandatalab.com/

Once downloaded, place the file at one of the following paths:

```
alc_files/alc.csv       ← CSV format (preferred)
alc_files/alc.xlsx      ← Excel format (fallback)
alc_files/alc.dta       ← Stata format (fallback)
```

Or set `ALC_URL` to a direct download link if one is available.

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
ALC_URL            — (optional) direct download URL
```

In [ ]:
import os
import io
import requests
import pandas as pd
import pyreadstat
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# ALC dataset from Carbone & Pellegata (ISPI / University of Milan)
# Project page: https://www.ispionline.it/en/publication/africa-leadership-change-project-21287
# DataLab:      https://alcafricandatalab.com/
# Set ALC_URL to a direct download link if available, or place the file locally.
ALC_URL = os.getenv("ALC_URL", "")

# Local fallback: check for CSV, Excel, or Stata formats
ALC_LOCAL_DIR  = Path("alc_files")
ALC_LOCAL_CSV  = ALC_LOCAL_DIR / "alc.csv"
ALC_LOCAL_XLSX = ALC_LOCAL_DIR / "alc.xlsx"
ALC_LOCAL_DTA  = ALC_LOCAL_DIR / "alc.dta"

local_exists = ALC_LOCAL_CSV.exists() or ALC_LOCAL_XLSX.exists() or ALC_LOCAL_DTA.exists()

print(f"Run date       : {RUN_DATE}")
print(f"ALC_URL set    : {bool(ALC_URL)}")
print(f"Local files    : csv={ALC_LOCAL_CSV.exists()}, xlsx={ALC_LOCAL_XLSX.exists()}, dta={ALC_LOCAL_DTA.exists()}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load ALC data

Tries URL → CSV local → Excel local → Stata local.

In [ ]:
def load_alc_from_url(url: str) -> pd.DataFrame | None:
    if not url:
        return None
    print(f"Downloading ALC from {url} ...")
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Download failed: {e}")
        return None

    content_type = resp.headers.get("content-type", "")
    content = resp.content

    if "csv" in content_type or url.lower().endswith(".csv"):
        return pd.read_csv(io.BytesIO(content), low_memory=False)
    elif "excel" in content_type or url.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(content))
    elif url.lower().endswith(".dta"):
        df, _ = pyreadstat.read_dta(io.BytesIO(content))
        return df
    else:
        # Attempt CSV by default
        try:
            return pd.read_csv(io.BytesIO(content), low_memory=False)
        except Exception:
            return None


def load_alc_from_local() -> pd.DataFrame | None:
    if ALC_LOCAL_CSV.exists():
        print(f"Loading ALC from CSV: {ALC_LOCAL_CSV}")
        return pd.read_csv(ALC_LOCAL_CSV, low_memory=False)
    if ALC_LOCAL_XLSX.exists():
        print(f"Loading ALC from Excel: {ALC_LOCAL_XLSX}")
        return pd.read_excel(ALC_LOCAL_XLSX)
    if ALC_LOCAL_DTA.exists():
        print(f"Loading ALC from Stata: {ALC_LOCAL_DTA}")
        try:
            df, _ = pyreadstat.read_dta(str(ALC_LOCAL_DTA))
            return df
        except Exception as e:
            print(f"  pyreadstat failed ({e}); trying pandas")
            return pd.read_stata(str(ALC_LOCAL_DTA))
    return None


df_alc_raw = load_alc_from_url(ALC_URL)

if df_alc_raw is None:
    df_alc_raw = load_alc_from_local()

if df_alc_raw is None:
    print(
        "\nWARNING: ALC data not available.\n"
        "  The Africa Leadership Change dataset is provided by Carbone & Pellegata\n"
        "  (ISPI / University of Milan). Download it from:\n"
        "    https://www.ispionline.it/en/publication/africa-leadership-change-project-21287\n"
        "  Or visit the DataLab at: https://alcafricandatalab.com/\n"
        "\n"
        "  Once downloaded, place the file at one of:\n"
        f"    {ALC_LOCAL_CSV}\n"
        f"    {ALC_LOCAL_XLSX}\n"
        f"    {ALC_LOCAL_DTA}\n"
        "  Or set the ALC_URL environment variable to a direct download link.\n"
    )
else:
    df_alc_raw.columns = [c.lower().strip() for c in df_alc_raw.columns]
    print(f"ALC raw: {len(df_alc_raw):,} leader-spells | columns: {list(df_alc_raw.columns)}")

## Schema normalisation

ALC column names vary across versions. We map to a canonical schema,
tolerating column-name differences between dataset editions.

In [ ]:
# Canonical column name → list of possible source names (in preference order)
COL_ALIASES = {
    "country":         ["country", "countryname", "country_name", "statename"],
    "ccode":           ["ccode", "cow", "cow_code", "countrycode"],
    "leader":          ["leader", "leadername", "name"],
    "startdate":       ["startdate", "start_date", "entrydate", "entry_date"],
    "enddate":         ["enddate", "end_date", "exitdate", "exit_date"],
    "start_year":      ["start_year", "startyear", "entry_year", "entryyear", "year_start"],
    "end_year":        ["end_year",   "endyear",   "exit_year",  "exityear",  "year_end"],
    "entry_type":      ["entry_type", "entrytype", "how_entry", "entry", "method_entry"],
    "exit_type":       ["exit_type",  "exittype",  "how_exit",  "exit",  "method_exit"],
    "irregular_entry": ["irregular_entry", "irr_entry", "irreg_entry"],
    "irregular_exit":  ["irregular_exit",  "irr_exit",  "irreg_exit", "irregular"],
    "military_regime": ["military_regime", "milreg", "military", "mil_regime"],
}

if df_alc_raw is not None:
    df_alc = df_alc_raw.copy()
    rename_map = {}
    for canonical, aliases in COL_ALIASES.items():
        if canonical in df_alc.columns:
            continue  # already present
        for alias in aliases:
            if alias in df_alc.columns:
                rename_map[alias] = canonical
                break
    if rename_map:
        df_alc.rename(columns=rename_map, inplace=True)
        print(f"Renamed columns: {rename_map}")

    # Parse date columns if present
    for dcol in ["startdate", "enddate"]:
        if dcol in df_alc.columns:
            df_alc[dcol] = pd.to_datetime(df_alc[dcol], errors="coerce")

    # Derive year columns from dates if not already present
    if "start_year" not in df_alc.columns and "startdate" in df_alc.columns:
        df_alc["start_year"] = df_alc["startdate"].dt.year.astype("Int64")
    if "end_year" not in df_alc.columns and "enddate" in df_alc.columns:
        df_alc["end_year"] = df_alc["enddate"].dt.year.astype("Int64")

    # Cast numeric columns
    for icol in ["ccode", "start_year", "end_year", "irregular_entry",
                 "irregular_exit", "military_regime"]:
        if icol in df_alc.columns:
            df_alc[icol] = pd.to_numeric(df_alc[icol], errors="coerce").astype("Int64")

    print(f"ALC normalised: {len(df_alc):,} rows | columns: {list(df_alc.columns)}")
    df_alc.head(3)
else:
    df_alc = pd.DataFrame()

In [ ]:
if not df_alc.empty:
    write_parquet(df_alc, f"raw/alc/{RUN_DATE}/alc_spells.parquet")

## Derive country-year panel

In [ ]:
if not df_alc.empty and "start_year" in df_alc.columns:
    records = []

    for _, row in df_alc.iterrows():
        sy = row.get("start_year")
        ey = row.get("end_year")
        if pd.isna(sy):
            continue
        if pd.isna(ey):
            ey = sy
        sy, ey = int(sy), int(ey)

        for yr in range(sy, ey + 1):
            records.append({
                "ccode":           row.get("ccode"),
                "country":         row.get("country"),
                "year":            yr,
                "leader":          row.get("leader"),
                "is_entry_year":   int(yr == sy),
                "irregular_entry": row.get("irregular_entry", 0),
                "irregular_exit":  row.get("irregular_exit", 0),
                "military_regime": row.get("military_regime", 0),
            })

    df_exp = pd.DataFrame(records)

    group_cols = [c for c in ["ccode", "country", "year"] if c in df_exp.columns]

    df_alc_cy = (
        df_exp
        .groupby(group_cols, as_index=False)
        .agg(
            n_leaders             =("leader",          "nunique"),
            is_leader_change      =("is_entry_year",   "max"),
            irregular_entry_count =("irregular_entry", "sum"),
            irregular_exit_count  =("irregular_exit",  "sum"),
            any_military_regime   =("military_regime", "max"),
        )
    )

    for col in ["n_leaders", "is_leader_change", "irregular_entry_count",
                "irregular_exit_count", "any_military_regime"]:
        if col in df_alc_cy.columns:
            df_alc_cy[col] = df_alc_cy[col].astype("Int64")

    print(f"ALC country-year panel: {len(df_alc_cy):,} rows")
    print(f"Years: {df_alc_cy['year'].min()}–{df_alc_cy['year'].max()}")
    df_alc_cy.head(3)
else:
    df_alc_cy = pd.DataFrame()

In [ ]:
if not df_alc_cy.empty:
    write_parquet(df_alc_cy, f"raw/alc/{RUN_DATE}/alc_country_year.parquet")

## Summary

In [ ]:
print("=" * 55)
print("Africa Leadership Change (ALC) pull complete")
print("=" * 55)
if not df_alc.empty:
    print(f"  Leader spells   : {len(df_alc):,}")
if not df_alc_cy.empty:
    print(f"  Country-years   : {len(df_alc_cy):,}")
    print(f"  Year range      : {df_alc_cy['year'].min()}–{df_alc_cy['year'].max()}")
    if 'country' in df_alc_cy.columns:
        print(f"  Countries       : {df_alc_cy['country'].nunique()}")
print()
print("ADLS paths written:")
print(f"  raw/alc/{RUN_DATE}/alc_spells.parquet")
print(f"  raw/alc/{RUN_DATE}/alc_country_year.parquet")